In [ ]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np
import pickle

LOG.propagate = False

ble = get_ble_controller()
ble.connect()

In [ ]:
step_msgs = []

def step_data_handler(uuid, byte_array):
    s = ble.bytearray_to_string(byte_array)
    step_msgs.append(s)
    print(s)

ble.start_notify(ble.uuid['RX_STRING'], step_data_handler)

ble.send_command(CMD.START_STEP_RESPONSE, "130|1000")

In [ ]:
ble.send_command(CMD.SEND_STEP_RESPONSE, "")

In [40]:
ble.stop_notify(ble.uuid['RX_STRING'])

In [ ]:
t_list = []          # ms
tof_list = []        # mm
base_pwm_list = []
left_pwm_list = []
right_pwm_list = []

for msg in step_msgs:
    if msg.startswith("STEP,"):
        parts = msg.split(",")

        t_list.append(float(parts[1]))
        tof_list.append(float(parts[2]))
        base_pwm_list.append(float(parts[3]))
        left_pwm_list.append(float(parts[4]))
        right_pwm_list.append(float(parts[5]))

t_ms = np.array(t_list)
t = t_ms / 1000.0
tof_mm = np.array(tof_list)
base_pwm = np.array(base_pwm_list)
left_pwm = np.array(left_pwm_list)
right_pwm = np.array(right_pwm_list)

print("N =", len(t))
print("t range =", t[0], "to", t[-1], "s")
print("tof range =", tof_mm[0], "to", tof_mm[-1], "mm")

In [36]:
step_data = {
    "t_ms": t_ms,
    "t_s": t,
    "tof_mm": tof_mm,
    "base_pwm": base_pwm,
    "left_pwm": left_pwm,
    "right_pwm": right_pwm,
    "raw_msgs": step_msgs,
    "config": {
        "base_pwm_cmd": 130,
        "stop_dist_mm": 1000
    }
}

with open("lab7_step_response_130_1000_test1.pkl", "wb") as f:
    pickle.dump(step_data, f)

print("Saved to lab7_step_response_130_1000.pkl")

Saved to lab7_step_response_130_1000.pkl


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- 1) ToF vs time ----------
plt.figure(figsize=(7, 4))
plt.plot(t, tof_mm, marker='o', label='ToF')
plt.xlabel("Time (s)")
plt.ylabel("Distance (mm)")
plt.title("ToF Sensor Output")
plt.grid(True)
plt.legend()
plt.show()


# ---------- 2) Computed speed vs time ----------
dt = np.diff(t)
d_tof = np.diff(tof_mm)
v_m_s = (d_tof / 1000.0) / dt   # mm/s -> m/s
t_v = 0.5 * (t[:-1] + t[1:])

plt.figure(figsize=(7, 4))
plt.plot(t_v, v_m_s, marker='o', label='Computed speed')
plt.xlabel("Time (s)")
plt.ylabel("Speed (m/s)")
plt.title("Computed Speed from ToF")
plt.grid(True)
plt.legend()
plt.show()


# ---------- 3) Motor input vs time ----------
plt.figure(figsize=(7, 4))
plt.plot(t, base_pwm, marker='o', label='Base PWM')
plt.plot(t, left_pwm, marker='o', label='Left PWM')
plt.plot(t, right_pwm, marker='o', label='Right PWM')
plt.xlabel("Time (s)")
plt.ylabel("PWM")
plt.title("Motor Input")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

# 读取 pickle
with open("lab7_step_response_130_1000_test1.pkl", "rb") as f:
    step_data = pickle.load(f)

# 取出数据
t_ms = step_data["t_ms"]
t = step_data["t_s"]
tof_mm = step_data["tof_mm"]
base_pwm = step_data["base_pwm"]
left_pwm = step_data["left_pwm"]
right_pwm = step_data["right_pwm"]

# 1) ToF vs time
plt.figure(figsize=(7,4))
plt.plot(t, tof_mm, marker='o', label='ToF')
plt.xlabel("Time (s)")
plt.ylabel("Distance (mm)")
plt.title("ToF Sensor Output")
plt.grid(True)
plt.legend()
plt.show()

# 2) Speed vs time
dt = np.diff(t)
d_tof = np.diff(tof_mm)
v_m_s = (d_tof / 1000.0) / dt
t_v = 0.5 * (t[:-1] + t[1:])

plt.figure(figsize=(7,4))
plt.plot(t_v, v_m_s, marker='o', label='Computed speed')
plt.xlabel("Time (s)")
plt.ylabel("Speed (m/s)")
plt.title("Computed Speed from ToF")
plt.grid(True)
plt.legend()
plt.show()

# 3) Motor input
plt.figure(figsize=(7,4))
plt.plot(t, base_pwm, marker='o', label='Base PWM')
plt.plot(t, left_pwm, marker='o', label='Left PWM')
plt.plot(t, right_pwm, marker='o', label='Right PWM')
plt.xlabel("Time (s)")
plt.ylabel("PWM")
plt.title("Motor Input")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import pickle

with open("lab7_step_response_130_1000_test1.pkl", "rb") as f:
    step_data = pickle.load(f)

print(step_data.keys())

t_ms = step_data["t_ms"]
t = step_data["t_s"]
tof_mm = step_data["tof_mm"]
base_pwm = step_data["base_pwm"]
left_pwm = step_data["left_pwm"]
right_pwm = step_data["right_pwm"]

print("t_ms =", t_ms)
print("t =", t)
print("tof_mm =", tof_mm)
print("base_pwm =", base_pwm)
print("left_pwm =", left_pwm)
print("right_pwm =", right_pwm)

In [ ]:
import numpy as np

# 已有: t, tof_mm

# 1) 5-point moving average on ToF
window = 5
t_sm = np.convolve(t, np.ones(window)/window, mode='valid')
tof_sm = np.convolve(tof_mm, np.ones(window)/window, mode='valid')

# 2) Differentiate smoothed ToF -> speed
v_sm = (np.diff(tof_sm) / 1000.0) / np.diff(t_sm)   # m/s
t_v_sm = 0.5 * (t_sm[:-1] + t_sm[1:])

# 3) steady-state speed = average magnitude of last 8 points
v_ss = np.mean(np.abs(v_sm[-8:]))

# 4) 90% target
v_90 = 0.9 * v_ss

# 5) first sustained crossing: 3 consecutive points above threshold
cross_idx = None
above = np.abs(v_sm) >= v_90
for i in range(len(above) - 2):
    if above[i:i+3].all():
        cross_idx = i
        break

t_90 = t_v_sm[cross_idx]
v_at_90 = abs(v_sm[cross_idx])

print("v_ss =", v_ss, "m/s")
print("0.9*v_ss =", v_90, "m/s")
print("t_90 =", t_90, "s")
print("v(t_90) =", v_at_90, "m/s")

In [ ]:
import numpy as np

# 你前面已经算出来的值
v_ss = 2.2807626251379145   # m/s
t_90 = 1.3546               # s
v_at_90 = 2.085858585858568 # m/s

# 推荐：用 normalized input u = 1
u_ss = 1.0

# 转成 mm/s，与后面距离单位一致
v_ss_mm_s = v_ss * 1000.0

d = u_ss / v_ss_mm_s
m = -d * t_90 / np.log(0.1)

print("v_ss =", v_ss, "m/s")
print("t_90 =", t_90, "s")
print("v(t_90) =", v_at_90, "m/s")
print("d =", d)
print("m =", m)

In [ ]:
import numpy as np

# ----- From Part 1 -----
d = 4.3844983646184195e-4
m = 2.579379803501347e-4

# sampling time from your logged data
Delta_T = np.mean(np.diff(t))   # t is in seconds

# state dimension
n = 2

# continuous-time matrices
A = np.array([
    [0, 1],
    [0, -d/m]
])

B = np.array([
    [0],
    [1/m]
])

# discretized matrices
Ad = np.eye(n) + Delta_T * A
Bd = Delta_T * B

print("Delta_T =", Delta_T)
print("A =\n", A)
print("B =\n", B)
print("Ad =\n", Ad)
print("Bd =\n", Bd)

In [ ]:
import numpy as np

# Measurement matrix
C = np.array([[-1, 0]])

# Initial state vector: [position; velocity]
# Use negative distance to match the course convention
x = np.array([
    [-tof_mm[0]],
    [0]
])

print("C =\n", C)
print("x =\n", x)

In [ ]:
import numpy as np

# ----- Initial state uncertainty -----
sigma_x0 = 100.0   # mm
sigma_v0 = 500.0   # mm/s

Sigma = np.array([
    [sigma_x0**2, 0],
    [0, sigma_v0**2]
])

# ----- Process noise -----
sigma_1 = 50.0     # mm
sigma_2 = 300.0    # mm/s

Sigma_u = np.array([
    [sigma_1**2, 0],
    [0, sigma_2**2]
])

# ----- Measurement noise -----
sigma_3 = 40.0     # mm

Sigma_z = np.array([[sigma_3**2]])

print("Sigma =\n", Sigma)
print("Sigma_u =\n", Sigma_u)
print("Sigma_z =\n", Sigma_z)

In [68]:
def kf(mu, sigma, u, y):
    # prediction
    mu_p = Ad.dot(mu) + Bd.dot(u)
    sigma_p = Ad.dot(sigma.dot(Ad.transpose())) + Sigma_u

    # update
    sigma_m = C.dot(sigma_p.dot(C.transpose())) + Sigma_z
    kkf_gain = sigma_p.dot(C.transpose()).dot(np.linalg.inv(sigma_m))

    y_m = y - C.dot(mu_p)
    mu = mu_p + kkf_gain.dot(y_m)
    sigma = (np.eye(2) - kkf_gain.dot(C)).dot(sigma_p)

    return mu, sigma

In [ ]:
mu = x.copy()
sigma = Sigma.copy()

kf_dist_mm = [-mu[0, 0]]   
kf_vel_mm_s = [mu[1, 0]]

for i in range(1, len(tof_mm)):
    u_scaled = np.array([[base_pwm[i] / 130.0]])  
    y = np.array([[tof_mm[i]]])                   

    mu, sigma = kf(mu, sigma, u_scaled, y)

    kf_dist_mm.append(-mu[0, 0])   #
    kf_vel_mm_s.append(mu[1, 0])

kf_dist_mm = np.array(kf_dist_mm)
kf_vel_mm_s = np.array(kf_vel_mm_s)

print("kf_dist_mm shape =", kf_dist_mm.shape)
print("kf_vel_mm_s shape =", kf_vel_mm_s.shape)
print("first 5 kf_dist_mm =", kf_dist_mm[:5])
print("last 5 kf_dist_mm =", kf_dist_mm[-5:])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
plt.plot(t, tof_mm, marker='o', label='ToF')
plt.plot(t, kf_dist_mm, marker='o', label='KF')
plt.xlabel("Time (s)")
plt.ylabel("Distance (mm)")
plt.title("KF Output vs. ToF Data")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(t, kf_vel_mm_s / 1000.0, marker='o', label='KF velocity')
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.title("KF Estimated Velocity")
plt.grid(True)
plt.legend()
plt.show()

In [72]:
final_kf_params = {
    "Delta_T": Delta_T,
    "A": A,
    "B": B,
    "Ad": Ad,
    "Bd": Bd,
    "C": C,
    "x0": x,
    "Sigma": Sigma,
    "Sigma_u": Sigma_u,
    "Sigma_z": Sigma_z
}

import pickle
with open("lab7_final_kf_params.pkl", "wb") as f:
    pickle.dump(final_kf_params, f)

print("Saved lab7_final_kf_params.pkl")

Saved lab7_final_kf_params.pkl
